In [0]:
bronze_df = spark.readStream.table("workspace.default.bronze_orders")

In [0]:
from pyspark.sql.functions import col, expr, to_timestamp

silver_df = (
    bronze_df
    .filter("price > 0 AND quantity > 0")        # remove bad records
    .dropDuplicates(["order_id"])                # remove duplicates
    .withColumn("total_amount", expr("price * quantity"))
    .withColumn("order_time", to_timestamp("timestamp"))
)

In [0]:
silver_df.writeStream \
  .format("delta") \
  .trigger(availableNow=True) \
  .option(
      "checkpointLocation",
      "/Volumes/workspace/default/etl_volume/checkpoints/bronze_silver"
  ) \
  .toTable("workspace.default.silver_orders_etl")

In [0]:
%sql
SHOW TABLES IN workspace.default;